In [1]:
import numpy as np
import scipy as sp
import pandas as pd
import matplotlib.pyplot as plt
import datetime
from OSDE.LegendreExpSPDensity import LegExp, LegExpSPDensity, LegExpResult
from StocProcess.RBM import RBMTransProb, MakeRBMTransProbFunc
from QAE.RQAE import RQAE

In [2]:
# RBM parameters
c = -1
d = 1
x0 = 0.5 * (c + d)
t0 = 0
mu = 0.5
sigma = 1.0
n_terms = 5

# approximation setting
maxDeg = 5
R = 12
K0 = 6

In [3]:
Ns = (2 ** np.linspace(3, 6.5, 8)).astype(int)
print(Ns)

[ 8 11 16 22 32 45 64 90]


In [4]:
Ks = []
pEsts = []
totalQueryNums = []
maxDepths = []

for N in Ns:
    legExpResultPrev = None
    ts = np.concatenate([[0], np.linspace(0.2, 0.6, N)])
    K = int(np.ceil(K0 + np.log2(N**0.5)))
    Ks.append(K)
    totalQueryNum = 0
    maxDepth = 0
    print(datetime.datetime.now(), "N=", N, "K=", K)

    for i in range(len(ts)-1):
        print("i_t=", i, datetime.datetime.now())
        transProbFunc = MakeRBMTransProbFunc(ts[i+1], ts[i], c, d, mu, sigma, n_terms)

        if i == 0:
            densFunc = lambda x: transProbFunc(x, x0)
            legExpResult = LegExp(densFunc, maxDeg)
        else:
            legExpResult = LegExpSPDensity(legExpResultPrev.fApp, transProbFunc, maxDeg)

        coefs = np.zeros(maxDeg+1)
        coefs[0] = 0.5

        for l in range(1, maxDeg+1):
            a = 0.5 * (legExpResult.coefs[l] / (l + 0.5) + 1)
            rqaeResult = RQAE(a, K, R)
            coefs[l] = (2 * rqaeResult.aEst -1) * (l + 0.5)
            totalQueryNum += rqaeResult.TotalQueryNum
            maxDepth = max(maxDepth, rqaeResult.MaxDepth)

        legExpResultPrev = LegExpResult(coefs)

    pEsts.append(sp.integrate.quad(legExpResultPrev.fApp, x0, 1.0)[0])
    totalQueryNums.append(totalQueryNum)
    maxDepths.append(maxDepth)

2025-01-27 21:57:58.814290 N= 8 K= 8
i_t= 0 2025-01-27 21:57:58.814290


c:\Users\koich\Desktop\Code\DivQCOSDE\QAE\MaximizeL.py:11: RuntimeWarning: divide by zero encountered in log
  neglogL = lambda theta: -np.dot(n1s, np.log(np.sin(thetaMuls * theta)**2)) - np.dot(n0s, np.log(np.cos(thetaMuls * theta)**2))


i_t= 1 2025-01-27 21:58:00.092715
i_t= 2 2025-01-27 21:58:50.862984
i_t= 3 2025-01-27 21:59:36.198076
i_t= 4 2025-01-27 22:00:01.902691
i_t= 5 2025-01-27 22:00:22.010568
i_t= 6 2025-01-27 22:00:34.390024
i_t= 7 2025-01-27 22:00:59.348486
2025-01-27 22:01:18.411427 N= 11 K= 8
i_t= 0 2025-01-27 22:01:18.412387
i_t= 1 2025-01-27 22:01:19.541692
i_t= 2 2025-01-27 22:01:42.198980
i_t= 3 2025-01-27 22:02:10.290050
i_t= 4 2025-01-27 22:02:36.691310
i_t= 5 2025-01-27 22:02:56.501154
i_t= 6 2025-01-27 22:03:19.369200
i_t= 7 2025-01-27 22:03:35.154059
i_t= 8 2025-01-27 22:03:57.264788
i_t= 9 2025-01-27 22:04:19.511460
i_t= 10 2025-01-27 22:04:39.778032
2025-01-27 22:05:02.204568 N= 16 K= 8
i_t= 0 2025-01-27 22:05:02.204568
i_t= 1 2025-01-27 22:05:03.361003
i_t= 2 2025-01-27 22:05:44.320488
i_t= 3 2025-01-27 22:06:24.613945
i_t= 4 2025-01-27 22:07:04.767015
i_t= 5 2025-01-27 22:07:45.571753
i_t= 6 2025-01-27 22:08:25.687227
i_t= 7 2025-01-27 22:09:10.060243
i_t= 8 2025-01-27 22:09:48.183700
i_t= 

In [5]:
# PDF at time t
transProbFunc = MakeRBMTransProbFunc(ts[-1], t0, c, d, mu, sigma, n_terms)

# Prob(X > (c + d)/2)
integFunc = lambda x: transProbFunc(x, x0)
pTrue, _ = sp.integrate.quad(integFunc, x0, 1.0)

In [6]:
retDf = pd.DataFrame(dict(N=Ns,
                          pTrue=np.repeat(pTrue, len(Ns)),
                          K=Ks,
                          pEst=pEsts,
                          absErr=np.abs(np.array(pEsts) - pTrue),
                          totalQueryNum=totalQueryNums,
                          maxDepth=maxDepths))

In [7]:
retDf.to_csv('DivideRBM_RQAE.csv', index=False)

In [8]:
retDf

,N,pTrue,K,pEst,absErr,totalQueryNum,maxDepth
0,8,0.649605,8,0.648562,0.001043,181741,255
1,11,0.649605,8,0.650681,0.001076,248962,255
2,16,0.649605,8,0.643692,0.005912,363104,255
3,22,0.649605,9,0.646199,0.003406,1004738,511
4,32,0.649605,9,0.652293,0.002689,1462653,511
5,45,0.649605,9,0.651718,0.002114,2055887,511
6,64,0.649605,9,0.648312,0.001293,2933866,511
7,90,0.649605,10,0.651868,0.002264,8246577,1023
